In [1]:
from langchain_core.documents import Document
from langchain_core.tools import tool
from typing import List, Any
from langchain_core.retrievers import BaseRetriever
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# Re-rank 모델
# uv add sentence-transformers
rerank_model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
cross_reranker = CrossEncoderReranker(model=rerank_model, top_n=2)

# 웹 검색 리트리버 클래스 정의
class WebSearchRetriever(BaseRetriever):
    search_wrapper: DuckDuckGoSearchAPIWrapper

    def __init__(self, search_wrapper: DuckDuckGoSearchAPIWrapper, **kwargs: Any):
        super().__init__(search_wrapper=search_wrapper, **kwargs)

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        results = self.search_wrapper.results(query, max_results=10)
        if not results:
            return [Document(page_content="관련 정보를 찾을 수 없습니다.")]
        
        formatted_docs = []
        for result in results:
            doc = Document(
                page_content=f'<Document href="{result.get("link", "")}"/>\n{result.get("snippet", "")}\n</Document>',
                metadata={
                    "source": "web search", 
                    "url": result.get("link", ""), 
                    "title": result.get("title", "")
                }
            )
            formatted_docs.append(doc)
        return formatted_docs

@tool
def web_search(query: str, search_period: str = 'm') -> List[Document]:
    """
    웹 검색을 수행하고 결과를 Document 리스트 형태로 반환하는 함수.
    search_period: 검색 기간 (d: 1일, w: 1주, m: 1달, y: 1년)
    """
    # WebSearchRetriever를 기반으로 ContextualCompressionRetriever를 초기화합니다.
    ddg_search_wrapper = DuckDuckGoSearchAPIWrapper(time=search_period)
    web_retriever = ContextualCompressionRetriever(
        base_compressor=cross_reranker, 
        base_retriever=WebSearchRetriever(search_wrapper=ddg_search_wrapper), 
    )

    docs = web_retriever.invoke(query)

    if len(docs) > 0:
        return docs
    
    return [Document(page_content="관련 정보를 찾을 수 없습니다.")]

In [2]:
# 도구 목록을 정의 
tools = [web_search]

In [ ]:
query = "KT 소액 결제 해킹 사건의 원인과 대책?"

tools[0].invoke(query)

[Document(metadata={'source': 'web search', 'url': 'https://reasonablegift.tistory.com/175', 'title': 'Kt 소액결제 해킹 사건 정리 (2025) - 피해 현황·원인·보상 절차 완벽 가이드'}, page_content='<Document href="https://reasonablegift.tistory.com/175"/>\n2025년 8월 수도권에서 발생한 KT 소액결제 해킹 사건은 불법 초소형 기지국을 통한 역대급 해킹 피해 사례입니다. 피해 현황, 해킹 원인, KT 및 정부 대응, 보상 절차와 예방법까지 한눈에 정리했습니다.2025년 8월 말부터 수도권 일부 지역을 중심으로 KT 및 KT 망을 사용하는 알뜰폰 고객들에게 휴대폰 ...\n</Document>'),
 Document(metadata={'source': 'web search', 'url': 'https://m.blog.naver.com/candle-mind/224003542873', 'title': 'Kt 소액 결제 사이버 침해사고 정리 - 네이버 블로그'}, page_content='<Document href="https://m.blog.naver.com/candle-mind/224003542873"/>\n최근 ** KT 소액 결제 사이버 침해 사고**가 전국적으로 확산되며 큰 사회적 파장을 불러일으키고 있습니다. 단순한 보안 문제가 아니라, 통신망 자체가 범행에 악용되었을 가능성이 제기되며 국민적 불안이 커지고 있습니다. 이번 글에서는 사건의 경위, 해킹 방식, KT 와 정부의 대응을 정리해 보겠습니다.\n</Document>')]

In [4]:
query = "초전도체 기술의 최근 발전과 상업적 응용 가능성은?"

tools[0].invoke(query)

[Document(metadata={'source': 'web search', 'url': 'https://jusiknara.com/초전도체-관련주-대장주-미래-투자처로-주목받는-이/', 'title': '초전도체 관련주 대장주, 미래 투자처로 주목받는 이유'}, page_content='<Document href="https://jusiknara.com/초전도체-관련주-대장주-미래-투자처로-주목받는-이/"/>\n초전도체는 특정 온도 이하에서 저항이 0이 되는 독특한 물질로, 전류를 무한히 흐르게 할 수 있는 혁신적인 기술입니다. 이 현상은 전자가 특정한 집단 행동을 통해 이루어지며, 이는 전기 에너지를 효율적으로 전달할 수 있도록 돕습니다. 초전도체의 응용 분야는 광범위하며, 전력 전송, 자기 공명 영상 (MRI), 자기부상열차 등 다양한 산업에서 활용되고 있습니다. 특히 최근의 기술 발전으로 인해 양자 컴퓨터와 같은 첨단 기술에서도 필수적인 요소로 자리잡고 있습니다.\n</Document>'),
 Document(metadata={'source': 'web search', 'url': 'https://www.yeolmupapago.com/2025/09/blog-post.html', 'title': '초전도체 연구 최신 동향: 상온 초전도체, 산업에 어떤 파장을 ...'}, page_content='<Document href="https://www.yeolmupapago.com/2025/09/blog-post.html"/>\n현재 초전도체 연구의 최신 동향은 어떤지, 특히 상온 초전도체가 현실화되었을 때 우리 사회에 어떤 파급 효과를 미칠지 자세히 살펴보아요.\n</Document>')]